# Solution for exercise 6.11
***

In [1]:
#Importing packages
from ISLP import load_data
import pandas as pd
import numpy as np
import statsmodels.api as sm
import sklearn.linear_model as skl

In [2]:
#Importing data
Boston = load_data('Boston')
Boston.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,5.33,36.2


In [3]:
#Split dataframe
y = Boston['crim']
X = Boston.drop(columns = 'crim')

## a)
---

In [4]:
#Split data
rng = np.random.default_rng(1)
n = len(y)
train_idx = rng.choice(n, size = n // 2, replace = False)
test_idx = np.setdiff1d(np.arange(n), train_idx)

Xtrain, Xtest = X.iloc[train_idx], X.iloc[test_idx]
ytrain, ytest = y.iloc[train_idx], y.iloc[test_idx]

In [5]:
#OLS
Xtrain_c = sm.add_constant(Xtrain)
Xtest_c = sm.add_constant(Xtest)

ols = sm.OLS(ytrain, Xtrain_c).fit()
print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:                   crim   R-squared:                       0.465
Model:                            OLS   Adj. R-squared:                  0.438
Method:                 Least Squares   F-statistic:                     17.38
Date:                Fri, 15 May 2026   Prob (F-statistic):           1.38e-26
Time:                        21:09:37   Log-Likelihood:                -831.89
No. Observations:                 253   AIC:                             1690.
Df Residuals:                     240   BIC:                             1736.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         13.8213     10.311      1.340      0.1

In [6]:
#Stats
pred_ols = ols.predict(Xtest_c)
MSE_ols = ((ytest - pred_ols) ** 2).mean()

print(f"OLS test MSE: {MSE_ols:.2f}")

OLS test MSE: 40.22


In [7]:
#Ridge
alphas = np.logspace(-4, 4, 200)
ridge_cv = skl.RidgeCV(alphas = alphas)
ridge_cv.fit(Xtrain, ytrain)

print(f"Ridge optimal alpha: {ridge_cv.alpha_:.4f}")

Ridge optimal alpha: 107.1891


In [8]:
ridge = skl.Ridge(alpha = ridge_cv.alpha_)
ridge.fit(Xtrain, ytrain)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",107.18913192051286
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gr

In [10]:
pred_ridge = ridge.predict(Xtest)
MSE_ridge = ((ytest - pred_ridge) ** 2).mean()

print(f"Ridge test MSE: {MSE_ridge:.2f}")

Ridge test MSE: 40.51


In [12]:
#Lasso
lasso_cv = skl.LassoCV(alphas = alphas)
lasso_cv.fit(Xtrain, ytrain)

print(f"Lasso optimal alpha: {lasso_cv.alpha_:.2f}")

Lasso optimal alpha: 0.26


In [13]:
lasso = skl.Lasso(alpha = lasso_cv.alpha_)
lasso.fit(Xtrain, ytrain)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",np.float64(0.2612675225563329)
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [ ]:
#MSE
pred_lasso = lasso.predict(Xtest)
MSE_lasso = ((ytest - pred_lasso) ** 2).mean()

print(f"Lasso test MSE: {MSE_lasso:.2f}")

Lasso test MSE: 40.47


In [18]:
coef_df = pd.DataFrame({
    'OLS' : ols.params.values, 
    'Ridge' : np.concatenate([[ridge.intercept_], ridge.coef_]), 
    'Lasso' : np.concatenate([[lasso.intercept_], lasso.coef_])
}, index = ols.params.index)

print(coef_df.round(4))

             OLS   Ridge   Lasso
const    13.8213  6.6370  5.6753
zn        0.0490  0.0441  0.0473
indus    -0.1125 -0.1244 -0.1160
chas      0.1673 -0.0061  0.0000
nox      -8.8810 -0.0397  0.0000
rm        0.6322  0.1966  0.0000
age      -0.0064 -0.0038 -0.0018
dis      -1.1151 -0.6985 -0.6962
rad       0.5716  0.5338  0.5259
tax      -0.0030 -0.0022 -0.0021
ptratio  -0.3501 -0.2262 -0.1439
lstat     0.2550  0.2489  0.2414
medv     -0.2161 -0.1584 -0.1367


## b)
***

### Interpretation

Alle tre modeller gir omtrent samme test-MSE: OLS (40.22), Lasso (40.47) og Ridge (40.51). 
OLS er marginalt best, men forskjellene er svært små. Dette tyder på at modellen ikke 
overfittes særlig, og at regularisering ikke er nødvendig i dette tilfellet.

## c)
***

### Interpretation

Lasso bruker ikke alle variablene. chas, nox og rm ble satt til nøyaktig null, 
noe som betyr at de ikke bidrar til prediksjon av kriminalitet i Lasso-modellen. 
Dette skyldes at disse variablene er korrelert med andre prediktorer — særlig nox, 
som har en stor koeffisient i OLS (-8.88) men settes til null av Lasso. 
Lasso klarer å identifisere at den samme informasjonen fanges opp av andre variabler, 
og velger å ekskludere de overflødige. En enklere modell med 9 variabler gir 
altså omtrent like god prediksjonsevne som en full modell med 12.
